#Fase 2: Generación de datos textuales sintéticos (Prompting)

**Objetivo:** Generar un corpus de reseñas sintéticas aplicando técnicas de *prompting* sobre el modelo cuantizado seleccionado en la Fase 1 (Llama-3.2-3B-Instruct, GGUF) ejecutado en CPU, con el fin de analizar el efecto de cada técnica en **calidad**, **diversidad** y **coherencia**.

**Qué se hace en este notebook:**
1) Se carga el **modelo cuantizado** con `llama-cpp-python`.
2) Se define una **instrucción base** + **condiciones de contexto** (producto, perfil, valoración, tono) + **restricciones globales** (longitud, estilo, formato, prohibiciones).
3) Se implementan y ejecutan técnicas de *prompting*:
- **Técnicas directas** (una sola llamada al modelo por reseña):
  - `zero_shot`
  - `one_shot`
  - `few_shot`
  - `topic_controlled`
  - `randomized`
  - `zero_shot_topic_generation`
  - `instruction_based`
- **Técnicas multipaso** (varias etapas por reseña):
  - `iterative_prompting` (reescritura/refinamiento)
  - `feedback_driven_prompting` (evalúa y regenera si no cumple)
  - `automatic_prompt_engineering` (genera variantes de prompt y selecciona)

**Diseño experimental:**
- Técnicas directas: 20 condiciones × 5 muestras = 100 reseñas por técnica.
- Técnicas multipaso: 10 condiciones × 5 muestras = 50 reseñas por técnica (por coste computacional: varias llamadas por muestra).
- Se guarda, junto a cada reseña, el prompt, el tema (si aplica) y el nº de intentos/iteraciones.

**Salidas (archivos) generadas:**
- Por técnica: se guardan dos ficheros en la carpeta `outputs/`:
  - `<técnica>.csv`
  - `<técnica>.xlsx`
  y se descargan automáticamente en Colab.
- Dataset final unificado: `/content/dataset.xlsx`

Además, se muestran en pantalla tablas de resumen para cada bloque de métricas.

> Nota: el diseño busca comparabilidad entre técnicas (misma estructura de columnas y mismas restricciones globales).

## Paso 1: Instalación de librerías

En este notebook se usa `llama-cpp-python` para ejecutar el modelo cuantizado en formato **GGUF**.  

El resto de librerías se usan para:
- manipulación de datos (`pandas`)
- progreso en bucles (`tqdm`)
- utilidades (`os`, `re`, `json`, `random`, `typing`)

In [ ]:
# INSTALACIÓN DE LLAMA-CPP-PYTHON
# Esta librería permite ejecutar modelos cuantizados en formato GGUF de forma local (CPU)
# sin necesidad de GPU, lo que reduce significativamente los requisitos computacionales.

!pip install -U llama-cpp-python


## Paso 2: Carga del modelo

Se carga el modelo cuantizado (GGUF) desde Hugging Face.

**Parámetro clave:**
- `n_ctx=2048`: tamaño de contexto máximo (tokens de entrada + tokens generados).  
  Se fija en un valor intermedio para equilibrar capacidad vs consumo de memoria.

In [ ]:
# CARGA DEL MODELO CUANTIZADO
# Se carga Llama-3.2-3B-Instruct en formato GGUF (cuantizado a 4 bits) desde Hugging Face.

from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="bartowski/Llama-3.2-3B-Instruct-GGUF",
    filename="Llama-3.2-3B-Instruct-Q4_K_M.gguf",
    n_ctx=2048
)


## Paso 3: Generación de texto

A partir de aquí se generan reseñas sintéticas usando distintas técnicas de *prompting*.  

Se divide en dos bloques:
- **Técnicas directas**: una sola llamada al modelo por reseña.
  - Zero-shot
  - One-shot
  - Few-shot
  - Topic-controlled prompting
  - Randomized prompting
  - Zero-shot topic generation
  - Instruction based prompting
- **Técnicas multi-paso**: varias llamadas (p.ej. generación + reescritura / evaluación + corrección / búsqueda de prompts):
  - iterative prompting
  - feedback-driven prompting
  - automatic prompt engineering

### Técnicas de prompting directas

#### Elementos comunes

Estas 7 técnicas, tienen como base de funcionamiento los mismos elementos base, aunque no todas los usen todos. Estos elementos son:

**1. Intrucción base**

Es la misma para todas las técnicas ➡ *"Escribe una reseña textual realista, similar a la que redactaría un cliente real, sobre un producto de un pequeño negocio local."*

**2. Contexto**

Son las condiciones bajo las cuales se genera cada reseña. Tenemos un conjunto base de condiciones cerrado (producto, perfil de cliente, valoración y tono) y el tema de la reseña. El tema será implícito o explícito según la técnica.

**3. Demostraciones**

Son ejemplos reales de reseñas utilizados como guía en algunas técnicas. Estas no corresponden necesariamente al mismo producto o valoración que las condiciones de generación, ya que su función es transmitir patrones de estilo y estructura propios de reseñas reales, y no contenido específico.

Las demostraciones han sido extraidas de un dataset de reseñas de amazon de Kaggle ([texto del enlace](https://www.kaggle.com/datasets/kritanjalijain/amazon-reviews)). Se han seleccionado 5 reseñas de ejemplo de tonos variados y longitud similar a la que se pedirá al LLM. Originalmente estaban en inglés pero han sido traducidas al español.

**4. Restricciones goblales**

Son reglas formales que se aplican a todas las técnicas de forma implícita. Incluyen el número de palabras, el tipo de salida, el formato y exclusiones como no mencionar la palabra IA. Aseguran uniformaidad entre los resultados para poder compararlos entre sí.

> En las siguientes celdas se define:
> - una configuración general: longitud objetivo, nº de condiciones y muestras, parámetros de generación.
> - unas condiciones base (PRODUCT_CONDITIONS): lista de 20 combinaciones cerradas (product, persona, rating, tone).
> - un banco de temas (REVIEW_TOPICS): lista de 20 temas.
> - demostraciones para las técnicas one-shot y few-shot.
> - una serie de funciones comunes para construir prompts, generar texto, formar filas y guardar archivos.


In [ ]:
# IMPORTACIÓN DE LIBRERÍAS ESENCIALES

# Se importan las librerías necesarias para:
# - Manipulación de datos: pandas
# - Progreso en bucles: tqdm (visualización de barras de progreso)
# - Utilidades generales: os, re, json, random, datetime, typing

import os                  # Manejo del sistema de archivo y variables de entorno
import re                  # Expresiones regulares (procesamiento de texto)
import json                # Serialización de datos en JSON
import random              # Funciones aleatorias
from datetime import datetime  # Timestamps y fechas
from typing import Dict, Any, Optional, List, Tuple  # Type hints para functions (claridad de código)

import pandas as pd        # Manipulación de DataFrames
from tqdm import tqdm      # Barras de progreso en bucles


In [ ]:
# CONFIGURACIÓN GLOBAL DE PARÁMETROS

# Estos parámetros definen la estructura experimental y el comportamiento general
# del modelo durante la generación de reseñas sintéticas.

# Restricciones de longitud
MIN_WORDS = 90
MAX_WORDS = 130

# Configuración del dataset experimental. Total por técnica directa: 20 * 5 = 100 reseñas
N_CONDITIONS = 20
K_PER_CONDITION = 5

# Parámetros de generación del modelo Llama
GEN_DEFAULT = dict(
    max_tokens=220,     # Tokens máximos a generar (equivalente aproximado a ~150 palabras)
    temperature=0.8,    # Control de aleatoriedad
    top_p=0.9,          # Nucleus sampling (considera el 90% más probable de tokens)
                        # Mejora coherencia descartando colas muy improbables
)

# Configuración de output
OUT_DIR = "outputs"     # Carpeta donde se guardan CSV y XLSX de cada técnica
os.makedirs(OUT_DIR, exist_ok=True)  # Crear carpeta si no existe


In [ ]:
# BANCO DE DATOS: CONDICIONES Y TEMAS

# Aquí se definen el conjunto cerrado de condiciones de generación (producto + persona + rating + tono)
# y un banco de temas que se usarán en técnicas que requieren temática explícita.

# CONDICIONES BASE (20)
# Cada condición es una combinación cerrada de:
# - product: tipo de producto de negocio local
# - persona: perfil del cliente
# - rating: valoración (1-5 estrellas)
# - tone: tono esperado de la reseña (cercano, neutral, crítico, formal, informal)

PRODUCT_CONDITIONS = [
    {"product": "Cafetera italiana",        "persona": "persona que prepara café en casa a diario",          "rating": 5, "tone": "cercano"},
    {"product": "Sartén antiadherente",     "persona": "cocinero aficionado",                                "rating": 4, "tone": "neutral"},
    {"product": "Batidora de mano",         "persona": "persona que hace smoothies en casa",                 "rating": 3, "tone": "informal"},
    {"product": "Tostadora",                "persona": "persona con desayunos rápidos",                      "rating": 2, "tone": "crítico"},
    {"product": "Juego de tuppers",         "persona": "persona que lleva comida al trabajo",                "rating": 4, "tone": "cercano"},
    {"product": "Robot aspirador básico",   "persona": "hogar con mascota",                                  "rating": 2, "tone": "crítico"},
    {"product": "Escoba de microfibra",     "persona": "persona que limpia su piso semanalmente",            "rating": 4, "tone": "neutral"},
    {"product": "Organizador de armario",   "persona": "persona que busca orden en casa",                    "rating": 3, "tone": "formal"},
    {"product": "Lámpara de escritorio LED","persona": "estudiante universitario",                           "rating": 4, "tone": "neutral"},
    {"product": "Soporte para portátil",    "persona": "teletrabajador",                                     "rating": 5, "tone": "formal"},
    {"product": "Ratón ergonómico",         "persona": "persona que pasa muchas horas frente al ordenador",  "rating": 3, "tone": "neutral"},
    {"product": "Cuaderno",                 "persona": "estudiante que toma apuntes a mano",                 "rating": 4, "tone": "informal"},
    {"product": "Botella",                  "persona": "persona que va a la oficina",                       "rating": 5, "tone": "cercano"},
    {"product": "Mochila urbana",           "persona": "viajero de fin de semana",                           "rating": 4, "tone": "neutral"},
    {"product": "Power bank portátil",      "persona": "persona que viaja con frecuencia",                  "rating": 2, "tone": "crítico"},
    {"product": "Secador de pelo",          "persona": "persona que se arregla por las mañanas",            "rating": 4, "tone": "neutral"},
    {"product": "Crema hidratante",         "persona": "persona con piel sensible",                         "rating": 5, "tone": "cercano"},
    {"product": "Cepillo de pelo",          "persona": "persona que cuida su cabello a diario",             "rating": 3, "tone": "formal"},
    {"product": "Altavoz Bluetooth",        "persona": "persona que escucha música en casa",                "rating": 4, "tone": "cercano"},
    {"product": "Manta para sofá",          "persona": "persona que busca confort en casa",                 "rating": 5, "tone": "cercano"},
]

# BANCO DE TEMAS (20)
# Lista de temas sobre los que podrían versar las reseñas.

REVIEW_TOPICS = [
    "experiencia de uso",
    "durabilidad",
    "calidad",
    "relación calidad-precio",
    "cumplimiento de expectativas",
    "diseño y aspecto general del producto",
    "comodidad",
    "funcionalidad frente a lo prometido",
    "nivel de satisfacción general",
    "aspectos que podrían mejorarse",
    "tamano y espacio",
    "sensación de durabilidad",
    "comparación con productos similares",
    "adecuación al estilo de vida del usuario",
    "atención al cliente",
    "experiencia de compra",
    "primeras impresiones frente a experiencia real",
    "envío",
    "grado de recomendación a otras personas",
    "valor aportado en el contexto personal"
]

# DEMOSTRACIONES (One-shot y Few-shot)
# Ejemplos reales de reseñas en español usados para guiar el estilo y estructura.
# Estos ejemplos NO son de los productos de las condiciones, su función es transmitir
# patrones estilísticos propios de reseñas reales de clientes.
# Origen: Dataset de Amazon Reviews (Kaggle), traducidas al español.

# ONE-SHOT: una única demostración para guiar formato y tono
DEMO_ONE_SHOT_ES = """
En primer lugar, me gustó el formato y el tono del libro (la forma en que la autora se dirige al lector).
Sin embargo, no sentí que aportara ninguno de los secretos internos que el libro prometía revelar.
Si estás empezando a informarte sobre la facultad de derecho y no conoces todos los requisitos de admisión,
entonces este libro puede ser de gran ayuda. Si ya has hecho tus deberes y estás buscando una ventaja adicional en el proceso de admisión,
recomiendo libros más específicos por tema. Por ejemplo, libros sobre cómo escribir tu declaración personal,
libros centrados específicamente en la preparación del LSAT (los libros de Powerscore fueron los más útiles para mí),
y también hay algunas páginas web con muy buenos consejos dirigidos a ayudar a las personas a las que vas a pedir cartas de recomendación.
Aun así, para quienes son nuevos en todo este proceso, este libro puede aclarar perfectamente los requisitos.
""".strip()

# FEW-SHOT: múltiples demostraciones (aquí 2 para no exceder límite de tokens del modelo)
# Se recortan las demostraciones para no saturar el contexto del prompt.
DEMOS_FEW_SHOT_ES = [
"""Mi querida Pat tiene una de las GRANDES voces de su generación. He escuchado este CD durante AÑOS y todavía ME ENCANTA.
Cuando estoy de buen humor me hace sentir aún mejor. Un mal humor simplemente se evapora como azúcar bajo la lluvia. Este CD rebosa VIDA.
Las voces son simplemente IMPRESIONANTES y las letras son demoledoras. Una de las joyas ocultas de la vida. Para mí, este es un CD de isla
desierta. Por qué nunca llegó a ser famosa está más allá de mi comprensión. Cada vez que lo pongo, da igual si son negros, blancos, jóvenes,
mayores, hombres o mujeres, TODO EL MUNDO dice lo mismo: "¿Quién estaba cantando?"
""".strip(),
"""
Compré este cargador en julio de 2003 y funcionó bien durante un tiempo. El diseño es bonito y cómodo. Sin embargo, después de aproximadamente un
año, las baterías ya no mantenían la carga. Casi que mejor comprar pilas alcalinas desechables, o buscar otro cargador que venga con baterías que
duren más.
""".strip()
]


In [ ]:
# FUNCIONES COMUNES - CONSTRUCCIÓN DE PROMPTS Y PROCESAMIENTO

# Este conjunto de funciones forma la base para construir prompts,
# generar texto, evaluar salidas y guardar resultados.

from google.colab import files

def constraints_block() -> str:
    """
    Bloque de restricciones globales aplicables a TODAS las técnicas de generación.
    Define reglas formales sobre longitud, formato, contenido y prohibiciones.

    Garantiza uniformidad en los requisitos para comparabilidad entre técnicas.
    """
    return f"""Restricciones globales:
- Longitud: {MIN_WORDS}-{MAX_WORDS} palabras.
- Contenido: Incluye detalles concretos sobre la experiencia de uso.
- Estilo: natural y humano, propio de un cliente real.
- Formato: un único párrafo, sin listas ni numeración.
- Prohibido: mencionar modelos de lenguaje, IA o sistemas automáticos.
- Salida: devuelve únicamente el texto final de la reseña.
"""

def base_instruction() -> str:
    """
    Instrucción base común a todas las técnicas.
    Define la tarea fundamental: escribir una reseña realista.
    """
    return "Escribe una reseña textual realista, similar a la que redactaría un cliente real, sobre un producto de un negocio local."

def instance_block(p: Dict[str, Any]) -> str:
    """
    Bloque de contexto/condiciones para una reseña específica.
    Contiene: producto, perfil del cliente, valoración y tono.

    Args:
        p: Diccionario con keys 'product', 'persona', 'rating', 'tone'

    Returns:
        Texto formateado con el contexto de la generación
    """
    return f"""Condiciones:
Producto: {p["product"]}
Perfil del cliente: {p["persona"]}
Valoración: {p["rating"]}/5
Tono: {p["tone"]}
"""

def condition_id(p: Dict[str, Any]) -> str:
    """
    Genera un ID único para cada condición (para rastreo).
    Combina: product | persona | rating | tone
    """
    return f'{p["product"]}|{p["persona"]}|{p["rating"]}|{p["tone"]}'

def chat_generate(prompt: str, gen_params: Dict[str, Any]) -> str:
    """
    Llama al modelo Llama para generar texto dado un prompt.

    Args:
        prompt: Texto del prompt a enviar al modelo
        gen_params: Diccionario con parámetros (max_tokens, temperature, top_p)

    Returns:
        Texto generado por el modelo (limpiado de saltos de línea múltiples)
    """
    # Usa el formato de conversación del modelo (role: user)
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        **gen_params
    )
    # Extrae el contenido y limpia saltos de línea múltiples
    text = out["choices"][0]["message"]["content"].strip()
    text = re.sub(r"\n{2,}", "\n", text).strip()
    return text

def make_row(sample_id: int,
             technique: str,
             p: Dict[str, Any],
             prompt: str,
             text: str,
             topic: Optional[str] = None,
             attempts: Optional[int] = None) -> Dict[str, Any]:
    """
    Construye una fila del DataFrame con toda la información de una reseña generada.

    Args:
        sample_id: ID único de la muestra
        technique: Nombre de la técnica usada (ej: 'zero_shot')
        p: Diccionario de condiciones (product, persona, rating, tone)
        prompt: Prompt enviado al modelo
        text: Reseña generada
        topic: Tema de la reseña (si aplica)
        attempts: Número de intentos/iteraciones (para técnicas multi-paso)

    Returns:
        Diccionario con todos los campos para una fila del DataFrame
    """
    return {
        "sample_id": sample_id,
        "technique": technique,
        "condition_id": condition_id(p),
        "product": p["product"],
        "persona": p["persona"],
        "rating": p["rating"],
        "tone": p["tone"],
        "topic": topic,
        "prompt": prompt,
        "text": text,
        "attempts": attempts,
    }

def save_outputs(df: pd.DataFrame, filename_base: str) -> None:
    """
    Guarda un DataFrame en formatos CSV y XLSX, y lo descarga desde Colab.

    Args:
        df: DataFrame con los datos generados
        filename_base: Nombre base del archivo (sin extensión)
                      Genera: {filename_base}.csv y {filename_base}.xlsx
    """
    # Rutas de salida
    csv_path = os.path.join(OUT_DIR, f"{filename_base}.csv")
    xlsx_path = os.path.join(OUT_DIR, f"{filename_base}.xlsx")

    # Guardar en ambos formatos
    df.to_csv(csv_path, index=False, encoding="utf-8")
    df.to_excel(xlsx_path, index=False)

    # Mostrar confirmación
    print("Guardado:")
    print("-", csv_path)
    print("-", xlsx_path)

    # Descargar automáticamente desde Colab
    files.download(csv_path)
    files.download(xlsx_path)

def quick_summary(df: pd.DataFrame, name: str) -> None:
    """
    Muestra un resumen rápido de un DataFrame.

    Args:
        name: Nombre de la técnica (para logging)
        df: DataFrame a resumir
    """
    print(f"\n{name} | n={len(df)}")


#### Técnicas básicas

**1. Zero-shot**

Esta técnica se usará como base ya que permite ver como se comporta el modelo con solo la tarea y el contexto mínimo. Sirve para medir la calidad base (qué tan realista es sin "ayuda"), la diversidad espontánea (si se repite el mismo patrón de prompt, ¿tiende a repetirse?) y el "sesgo natural" del modelo, es decir, en qué aspectos se fija sin nuestras indicaciones.

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: implícito.  
* Restricciones globales.

In [ ]:
def build_prompt_zero_shot(p: Dict[str, Any]) -> str:
    """
    Construye el prompt para zero-shot prompting.

    Bloques:
    - Instrucción base (qué hacer)
    - Contexto: producto + persona + rating + tono (el qué específico)
    - Restricciones globales (requisitos formales)
    - Tema: IMPLÍCITO (no se menciona)

    Args:
        p: Diccionario de condiciones (product, persona, rating, tone)

    Returns:
        Prompt formateado para enviar al modelo
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        "",
        constraints_block().strip()
    ]).strip()


def generate_zero_shot_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
) -> pd.DataFrame:
    """
    Genera dataset usando zero-shot prompting.

    Para cada condición de las N_CONDITIONS:
    - Genera K_PER_CONDITION reseñas
    - Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones (product + persona + rating + tone)
        k_per_condition: Número de muestras por condición
        gen_params: Parámetros de generación del modelo

    Returns:
        DataFrame con columnas: sample_id, technique, condition_id, product,
                                persona, rating, tone, topic, prompt, text, attempts
    """
    rows = []
    sample_id = 0

    # Itera sobre cada condición
    for p in tqdm(conditions, desc="Generando ZERO-SHOT"):
        # Construye el prompt una sola vez (es igual para todas las muestras de esta condición)
        prompt = build_prompt_zero_shot(p)

        # Genera k reseñas con el mismo contexto pero distinto output (por temperature)
        for _ in range(k_per_condition):
            # Llama al modelo
            text = chat_generate(prompt, gen_params)

            # Añade fila al dataframe
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="zero_shot",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=None,              # Técnica directa: sin tema explícito
                    attempts=1               # Una sola llamada al modelo
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con ZERO-SHOT...")
df_zero_shot = generate_zero_shot_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_zero_shot, "ZERO-SHOT")
save_outputs(df_zero_shot, "zero_shot")
print("\nPrimeras 5 filas:")
df_zero_shot.head()


**2. One-shot**

Con esta técnica vemos la capacidad de aprendizaje en contexto con un solo ejemplo. Sirve para observar si el modelo imita el formato y el estilo del ejemplo dado, si mejora la coherencia y realismo respecto al zero-shot y hasta qué punto sacrifica diversidad al anclarse a un único patrón.

En comparación con zero-shot suele generar textos más “correctos” y consistentes, pero existe el riesgo de que todas las reseñas se parezcan demasiado entre sí.

¿Mejora la calidad si muestro un único ejemplo representativo, sin llegar a sobrecondicionar al modelo?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: implícito.
* Demostraciones: una.
* Restricciones globales.

In [ ]:
def build_prompt_one_shot(p: Dict[str, Any], demo: str = DEMO_ONE_SHOT_ES) -> str:
    """
    Construye el prompt para one-shot prompting.

    Bloques:
    - Instrucción base
    - 1 demostración (guía para formato/estructura/estilo)
    - Contexto: producto + persona + rating + tono
    - Restricciones globales
    - Tema: IMPLÍCITO (no se menciona)

    Args:
        p: Diccionario de condiciones
        demo: Texto de ejemplo (una reseña real)

    Returns:
        Prompt formateado
    """
    return "\n".join([
        base_instruction(),
        "",
        "Ejemplo (para guiar estilo y estructura):",
        demo.strip(),
        "",
        instance_block(p).strip(),
        "",
        constraints_block().strip()
    ]).strip()


def generate_one_shot_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    demo: str = DEMO_ONE_SHOT_ES,
) -> pd.DataFrame:
    """
    Genera dataset usando one-shot prompting.

    Comparación con zero-shot:
    - Espera: mayor consistencia, mejor estructura, posible pérdida de diversidad

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo
        demo: Texto de demostración (ejemplo de reseña)

    Returns:
        DataFrame con las reseñas generadas
    """
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ONE-SHOT"):
        prompt = build_prompt_one_shot(p, demo=demo)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="one_shot",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=None,               # Sin tema explícito
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con ONE-SHOT...")
df_one_shot = generate_one_shot_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_one_shot, "ONE-SHOT")
save_outputs(df_one_shot, "one_shot")
print("\nPrimeras 5 filas:")
df_one_shot.head()


**3. Few-shot**

Con esta técnica vemos la capacidad del modelo para aprender patrones más estables cuando se le proporcionan varios ejemplos. Permite observar si aumenta la fidelidad (estructura, estilo, nivel de detalle), si las reseñas resultan menos genéricas y si aparece una pérdida de diversidad por repetición de patrones.

En comparación con one-shot suele mejorar la calidad media, pero incrementa el riesgo de outputs muy parecidos entre sí y dependencia excesiva del estilo de los ejemplos.

¿Hasta qué punto puedo mejorar la calidad del texto guiando al modelo con ejemplos, y a qué coste en diversidad?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: implícito.  
* Demostraciones: varias.
* Restricciones globales.

In [ ]:
def build_prompt_few_shot(p: Dict[str, Any], demos: List[str] = DEMOS_FEW_SHOT_ES) -> str:
    """
    Construye el prompt para few-shot prompting.

    Bloques:
    - Instrucción base
    - Varias demostraciones (múltiples ejemplos para guiar)
    - Contexto: producto + persona + rating + tono
    - Restricciones globales
    - Tema: IMPLÍCITO

    Args:
        p: Diccionario de condiciones
        demos: Lista de textos de ejemplo

    Returns:
        Prompt formateado
    """
    # Únsula los ejemplos separados por doble salto de línea
    demos_block = "\n\n".join([d.strip() for d in demos if d and d.strip()])

    return "\n".join([
        base_instruction(),
        "",
        "Ejemplos (para guiar estilo y estructura):",
        demos_block,
        "",
        instance_block(p).strip(),
        "",
        constraints_block().strip()
    ]).strip()


def generate_few_shot_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    demos: List[str] = DEMOS_FEW_SHOT_ES,
) -> pd.DataFrame:
    """
    Genera dataset usando few-shot prompting.

    Comparación con one-shot:
    - Espera: mayor fidelidad al estilo, posible uniformidad (menos diversidad)

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo
        demos: Lista de demostraciones

    Returns:
        DataFrame con las reseñas generadas
    """
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando FEW-SHOT"):
        prompt = build_prompt_few_shot(p, demos=demos)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="few_shot",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=None,               # Sin tema explícito
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con FEW-SHOT...")
df_few_shot = generate_few_shot_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_few_shot, "FEW-SHOT")
save_outputs(df_few_shot, "few_shot")
print("\nPrimeras 5 filas:")
df_few_shot.head()


Estas tres técnicas representan los enfoques básicos para la generación de texto con LLMs, diferenciándose principalmente por el nivel de información y ejemplos proporcionados al modelo. Estas estrategias permiten analizar el equilibrio entre diversidad y fidelidad en los datos sintéticos generados y constituyen el punto de partida para técnicas más avanzadas que introducen un mayor control sobre atributos específicos del texto.

#### Técnicas avanzadas

**4. Topic-controlled prompting**

Con esta técnica pasamos a mirar la capacidad del modelo para seguir una condición semántica explícita impuesta desde el prompt. Permite observar si el modelo centra el contenido de la reseña en el tema indicado, si mantiene coherencia temática a lo largo del texto y si es capaz de integrar el tema sin entrar en contradicción con el producto, el perfil del cliente, la valoración y el tono.

En comparación con zero-shot, one-shot y few-shot el foco del contenido ya no es decidido libremente por el modelo, reduciendo la variabilidad temática, pero aumenta el control sobre qué se está evaluando.

¿Hasta qué punto el modelo es capaz de generar reseñas coherentes cuando se le impone explícitamente el foco del contenido?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: explícito.  
* Restricciones globales.

In [ ]:
def build_prompt_topic_controlled(p: Dict[str, Any], topic: str) -> str:
    """
    Construye el prompt para topic-controlled prompting.

    Bloques:
    - Instrucción base
    - Contexto: producto + persona + rating + tono
    - Tema: EXPLÍCITO (se especifica directamente)
    - Restricciones globales

    Args:
        p: Diccionario de condiciones
        topic: Tema explícito de la reseña

    Returns:
        Prompt formateado
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema (explícito): {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_topic_controlled_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    topics: List[str] = REVIEW_TOPICS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
) -> pd.DataFrame:
    """
    Genera dataset con control temático.

    Asignación: condiciones[i] ↔ temas[i] (1:1 en orden)
    Requisito: len(conditions) == len(topics)

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        topics: Lista de temas (debe tener mismo tamaño que conditions)
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo

    Returns:
        DataFrame con las reseñas generadas

    Raises:
        ValueError: Si len(conditions) != len(topics)
    """
    # Validación: temas y condiciones deben coincidir en cantidad
    if len(conditions) != len(topics):
        raise ValueError(
            f"Topic-controlled requiere len(conditions) == len(topics). "
            f"Ahora: {len(conditions)} condiciones vs {len(topics)} temas."
        )

    rows = []
    sample_id = 0

    for i, p in enumerate(tqdm(conditions, desc="Generando TOPIC-CONTROLLED")):
        # Asigna el tema i-ésimo a la condición i-ésima
        topic = topics[i]
        prompt = build_prompt_topic_controlled(p, topic=topic)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="topic_controlled",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=topic,              # Tema explícito (asignado)
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con TOPIC-CONTROLLED...")
df_topic_controlled = generate_topic_controlled_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_topic_controlled, "TOPIC-CONTROLLED")
save_outputs(df_topic_controlled, "topic_controlled")
print("\nPrimeras 5 filas:")
df_topic_controlled.head()


**5. Randomized prompting**

Ahora ya no se evalúa únicamente la capacidad del modelo para cumplir una condición concreta, sino su capacidad para generar un conjunto diverso de reseñas manteniendo una calidad media estable. En concreto, se analiza el efecto de introducir variabilidad controlada en el proceso de generación y cómo esta variabilidad influye en la diversidad global del dataset, sin comprometer la coherencia interna ni el realismo de los textos generados.

A diferencia de topic-controlled prompting, el tema no se fija manualmente para cada generación, sino que se selecciona de forma aleatoria a partir de un conjunto predefinido, lo que permite aumentar la cobertura temática sin intervención manual directa.

¿Hasta qué punto es posible generar de forma automática un conjunto amplio y diverso de reseñas coherentes y realistas, manteniendo una calidad media aceptable bajo variación temática controlada?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: aleatorio dentro de un conjunto definido previamente.  
* Restricciones globales.

In [ ]:
def build_prompt_randomized(p: Dict[str, Any], topic: str) -> str:
    """
    Construye el prompt para randomized prompting.

    Similar a topic-controlled, pero el tema se selecciona aleatoriamente
    en lugar de asignarse manualmente.

    Bloques:
    - Instrucción base
    - Contexto: producto + persona + rating + tono
    - Tema: explícito pero ALEATORIO
    - Restricciones globales

    Args:
        p: Diccionario de condiciones
        topic: Tema seleccionado aleatoriamente

    Returns:
        Prompt formateado
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema (aleatorio): {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_randomized_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    topics: List[str] = REVIEW_TOPICS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Genera dataset con temas aleatorios.

    Para cada MUESTRA (no solo por condición), se elige aleatoriamente un tema
    del banco REVIEW_TOPICS. Esto aumenta la cobertura temática sin intervención manual.

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        topics: Banco de temas (la muestra elige aleatoriamente de aquí)
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo
        seed: Semilla para reproducibilidad del proceso aleatorio

    Returns:
        DataFrame con las reseñas generadas
    """
    # Inicializa propio RNG con semilla para reproducibilidad
    rng = random.Random(seed)

    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando RANDOMIZED"):
        for _ in range(k_per_condition):
            # Selecciona un tema aleatorio del banco
            topic = rng.choice(topics)
            prompt = build_prompt_randomized(p, topic=topic)

            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="randomized",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=topic,              # Tema seleccionado aleatoriamente
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con RANDOMIZED...")
df_randomized = generate_randomized_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_randomized, "RANDOMIZED")
save_outputs(df_randomized, "randomized")
print("\nPrimeras 5 filas:")
df_randomized.head()


**6. Zero-shot topic generation**

Con esta técnica el modelo genera temas relevantes a partir del contexto dado para cada generación. Sirve para evaluar la capacidad del modelo para identificar temas pertinentes para una reseña, si los temas generados son coherentes con el producto, el perfil del cliente, la valoración y el tono y si el uso de temas auto-generados permite aumentar la diversidad del contenido sin necesidad de definir previamente un espacio temático cerrado.

A diferencia de topic-controlled y randomized prompting el tema no es impuesto por el diseñador del prompt, ni se selecciona de forma aleatoria a partir de una lista, sino que emerge directamente del propio modelo en un paso previo a la generación de la reseña.

¿Hasta qué punto un LLM es capaz de proponer temas relevantes y generar reseñas coherentes a partir de ellos sin intervención humana directa en la selección temática?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: generado.  
* Restricciones globales.


In [ ]:
def build_prompt_topic_proposal(p: Dict[str, Any]) -> str:
    """
    PASO 1: Propone un tema breve y relevante condicionado al contexto.

    Parámetros de temperatura bajos para obtener salida limpia y determinista.
    Salida esperada: SOLO el tema (sin explicación ni comillas).

    Args:
        p: Diccionario de condiciones

    Returns:
        Prompt que pide generar un tema
    """
    return "\n".join([
        "Genera UN único tema breve y relevante para una reseña, coherente con las siguientes condiciones.",
        "Devuelve SOLO el tema (una frase corta), sin comillas, sin viñetas, sin explicación.",
        "",
        instance_block(p).strip(),
    ]).strip()


def clean_topic(raw: str) -> str:
    """
    Limpia la salida del modelo para extraer un tema usable.

    Acciones:
    - Toma la primera línea
    - Elimina comillas envolventes
    - Elimina prefijos típicos (si el modelo no obedece 100%)
    - Recorta espacios múltiples

    Args:
        raw: Texto bruto generado por el modelo

    Returns:
        Tema limpio y listo para usar
    """
    t = raw.strip().splitlines()[0].strip()  # Primera línea
    t = t.strip('"\'')                        # Elimina comillas

    # Elimina prefijos típicos que el modelo podría añadir
    t = re.sub(r"^(tema|tópico)\s*:\s*", "", t, flags=re.IGNORECASE).strip()

    # Recorta espacios múltiples
    t = re.sub(r"\s{2,}", " ", t).strip()

    return t


def build_prompt_zero_shot_topic_generation(p: Dict[str, Any], topic: str) -> str:
    """
    PASO 2: Genera la reseña usando el tema auto-generado.

    Bloques:
    - Instrucción base
    - Contexto: producto + persona + rating + tono
    - Tema: el tema generado en PASO 1
    - Restricciones globales

    Args:
        p: Diccionario de condiciones
        topic: Tema generado en el paso anterior

    Returns:
        Prompt para generar la reseña
    """
    return "\n".join([
        base_instruction(),
        "",
        instance_block(p).strip(),
        f"Tema (generado): {topic}",
        "",
        constraints_block().strip()
    ]).strip()


def generate_zero_shot_topic_generation_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    topic_gen_params: Optional[Dict[str, Any]] = None,
) -> pd.DataFrame:
    """
    Genera dataset con generación de temas en dos pasos.

    PASO 1: Generar tema (temperatura baja para determinismo)
    PASO 2: Generar reseña con ese tema (temperatura normal)

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo para la reseña
        topic_gen_params: Parámetros específicos para generar el tema
                         (defecto: temperatura más baja para estabilidad)

    Returns:
        DataFrame con las reseñas generadas
    """
    # Si no se especifican params para tema, usa los generales pero con temp más baja
    if topic_gen_params is None:
        topic_gen_params = dict(gen_params)
        topic_gen_params.update(dict(max_tokens=40, temperature=0.4, top_p=0.9))

    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ZERO-SHOT TOPIC GENERATION"):
        for _ in range(k_per_condition):
            # PASO 1: Proponer tema (con baja temperatura para consistencia)
            prompt_topic = build_prompt_topic_proposal(p)
            raw_topic = chat_generate(prompt_topic, topic_gen_params)
            topic = clean_topic(raw_topic)

            # PASO 2: Reseña con tema generado (temperatura normal)
            prompt_review = build_prompt_zero_shot_topic_generation(p, topic=topic)
            text = chat_generate(prompt_review, gen_params)

            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="zero_shot_topic_generation",
                    p=p,
                    prompt=prompt_review,
                    text=text,
                    topic=topic,              # Tema generado por el modelo
                    attempts=1                # Una iteración (2 llamadas = 1 iteración)
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con ZERO-SHOT TOPIC GENERATION (2 pasos)...")
df_zero_shot_topic_generation = generate_zero_shot_topic_generation_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_zero_shot_topic_generation, "ZERO-SHOT TOPIC GENERATION")
save_outputs(df_zero_shot_topic_generation, "zero_shot_topic_generation")
print("\nPrimeras 5 filas:")
df_zero_shot_topic_generation.head()


**One-shot topic generation** (Experimento adicional)

Se añade un ejemplo de tema en el PASO 1 para mejorar la calidad de los temas generados.

In [ ]:
DEMO_TOPIC_ONE_SHOT = "relación calidad-precio"  # Ejemplo de tema para guiar el formato

def build_prompt_topic_proposal_one_shot(
    p: Dict[str, Any],
    demo_topic: str = DEMO_TOPIC_ONE_SHOT
) -> str:
    """
    PASO 1 (ONE-SHOT): Propone un tema guiado con UN ejemplo.

    Se añade una demostración para que el modelo entienda mejor qué tipo de tema buscamos.

    Args:
        p: Diccionario de condiciones
        demo_topic: Texto de ejemplo (un tema representativo)

    Returns:
        Prompt que pide generar un tema con guía
    """
    return "\n".join([
        "Genera un único tema breve y relevante para una reseña, coherente con las siguientes condiciones.",
        "Devuelve solo el tema (una frase corta), sin comillas, sin viñetas, sin explicación.",
        "",
        "Ejemplo de tema:",
        demo_topic.strip(),
        "",
        instance_block(p).strip(),
    ]).strip()


def generate_one_shot_topic_generation_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    topic_gen_params: Optional[Dict[str, Any]] = None,
    demo_topic: str = DEMO_TOPIC_ONE_SHOT,
) -> pd.DataFrame:
    """
    Genera dataset con generación de temas one-shot (2 pasos).

    PASO 1: Generar tema con un ejemplo guía
    PASO 2: Generar reseña con ese tema

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo para la reseña
        topic_gen_params: Parámetros para generar temas (defecto: temperatura baja)
        demo_topic: Texto de ejemplo para guiar el tipo de tema

    Returns:
        DataFrame con las reseñas generadas
    """
    if topic_gen_params is None:
        # Temperatura más baja para mejora consistencia en generación de temas
        topic_gen_params = dict(gen_params)
        topic_gen_params.update(dict(max_tokens=40, temperature=0.4, top_p=0.9))

    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ONE-SHOT TOPIC GENERATION"):
        for _ in range(k_per_condition):
            # PASO 1: Proponer tema (one-shot con un ejemplo)
            prompt_topic = build_prompt_topic_proposal_one_shot(p, demo_topic=demo_topic)
            raw_topic = chat_generate(prompt_topic, topic_gen_params)
            topic = clean_topic(raw_topic)

            # PASO 2: Reseña con tema generado
            prompt_review = build_prompt_zero_shot_topic_generation(p, topic=topic)
            text = chat_generate(prompt_review, gen_params)

            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="one_shot_topic_generation",
                    p=p,
                    prompt=prompt_review,
                    text=text,
                    topic=topic,
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con ONE-SHOT TOPIC GENERATION (2 pasos)...")
df_one_shot_topic_generation = generate_one_shot_topic_generation_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_one_shot_topic_generation, "ONE-SHOT TOPIC GENERATION")
save_outputs(df_one_shot_topic_generation, "one_shot_topic_generation")
print("\nPrimeras 5 filas:")
df_one_shot_topic_generation.head()


No se aprecian mejoras

**7.  Instruction based prompting**

Con esta técnica se evalúa principalmente la capacidad del modelo para seguir instrucciones explícitas y estructuradas, más allá de condiciones aisladas como el tema. Permite observar si el modelo interpreta correctamente una instrucción descompuesta en partes claras (tarea, contexto, condiciones y formato), si mejora la alineación con los requisitos formales del texto, y si una formulación más estructurada del prompt reduce ambigüedades y resultados no deseados.

A diferencia de las técnicas anteriores no se introduce diversidad mediante ejemplos (one/few-shot), ni mediante variación o generación de temas (randomized o zero-shot topic generation), sino que el control se ejerce a través de la claridad y explicitud de la instrucción.

¿Hasta qué punto una formulación explícita y estructurada de la instrucción mejora la alineación del modelo con los requisitos de la tarea?

¿Que bloques usa?
* Instrucción base
* Contexto
     * producto + perfil del cliente + valoración + tono.
     * Tema: explícito.  
* Restricciones globales.

In [ ]:
def build_prompt_instruction_based(p: Dict[str, Any], topic: str) -> str:
    """
    Construye el prompt para instruction-based prompting.

    A diferencia de las técnicas anteriores, la instrucción se divide en secciones explícitas:
    - TAREA: qué hacer
    - CONTEXTO: condiciones específicas
    - FOCO: tema
    - REQUISITOS: restricciones formales

    Esta estructura explícita puede mejorar la claridad y alineación del modelo.

    Args:
        p: Diccionario de condiciones
        topic: Tema explícito (asignado 1:1 en orden)

    Returns:
        Prompt estructurado en secciones
    """
    return f"""TAREA
{base_instruction()}

CONTEXTO
{instance_block(p).strip()}

FOCO (TEMA)
{topic}

REQUISITOS
{constraints_block().strip()}
""".strip()


def generate_instruction_based_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    topics: List[str] = REVIEW_TOPICS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
) -> pd.DataFrame:
    """
    Genera dataset con instrucciones estructuradas.

    Asignación: condiciones[i] ↔ temas[i] (1:1 en orden, similar a topic-controlled)
    Requisito: len(conditions) == len(topics)

    Total: 20 condiciones × 5 muestras = 100 reseñas

    Args:
        conditions: Lista de condiciones
        topics: Lista de temas (debe tener mismo tamaño que conditions)
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo

    Returns:
        DataFrame con las reseñas generadas

    Raises:
        ValueError: Si len(conditions) != len(topics)
    """
    # Validación
    if len(conditions) != len(topics):
        raise ValueError(
            f"Instruction-based requiere len(conditions) == len(topics). "
            f"Ahora: {len(conditions)} condiciones vs {len(topics)} temas."
        )

    rows = []
    sample_id = 0

    for i, p in enumerate(tqdm(conditions, desc="Generando INSTRUCTION-BASED")):
        topic = topics[i]
        prompt = build_prompt_instruction_based(p, topic=topic)

        for _ in range(k_per_condition):
            text = chat_generate(prompt, gen_params)
            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="instruction_based",
                    p=p,
                    prompt=prompt,
                    text=text,
                    topic=topic,              # Tema asignado 1:1
                    attempts=1
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con INSTRUCTION-BASED...")
df_instruction_based = generate_instruction_based_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_instruction_based, "INSTRUCTION-BASED")
save_outputs(df_instruction_based, "instruction_based")
print("\nPrimeras 5 filas:")
df_instruction_based.head()


### Técnicas de prompting multi-paso

Las técnicas anteriores se basan en una única llamada de generación al modelo, variando el nivel de información, ejemplos o control semántico proporcionado en el prompt. Sin embargo, existen estrategias más avanzadas que descomponen el proceso de generación en múltiples pasos, permitiendo introducir mecanismos explícitos de evaluación y mejora progresiva del contenido generado. Estas estrategias se agrupan dentro de la denominada multi-step generation y permiten refinar la calidad, coherencia y diversidad del dataset sintético de forma iterativa.

#### Elementos comunes

In [ ]:
# CONFIGURACIÓN PARA TÉCNICAS MULTI-PASO

# Las técnicas multi-paso requieren múltiples llamadas al modelo, por lo que generan
# menos ejemplos por técnica (5 condiciones en lugar de 20, para viabilidad computacional).

MIN_WORDS = 90
MAX_WORDS = 130

# Dataset reducido (5 en lugar de 20 condiciones por eficiencia computacional)
# Las técnicas multi-paso hacen múltiples llamadas por muestra: generación + mejora/evaluación
N_CONDITIONS = 10    # Reducido para viabilidad
K_PER_CONDITION = 5  # Mismo número de muestras por condición

# Parámetros ajustados para mejor balance calidad-tokens durante iteraciones
GEN_DEFAULT = dict(
    max_tokens=220,       # Tokens máximos (equivalente a ~150 palabras)
    temperature=0.8,      # Mantenemos balance diversidad-coherencia
    top_p=0.9,
)


In [ ]:
# FUNCIONES AUXILIARES PARA TÉCNICAS MULTI-PASO

# Funciones de evaluación, validación y construcción de prompts compactos.

def word_count(text: str) -> int:
    """
    Cuenta el número de palabras en un texto usando expresión regular.

    Args:
        text: Texto a contar

    Returns:
        Número de palabras
    """
    return len(re.findall(r"\b\w+\b", text, flags=re.UNICODE))

def passes_basic_constraints(text: str) -> Tuple[bool, Dict[str, Any]]:
    """
    Evalúa si un texto cumple con las restricciones formales básicas.

    Verifica:
    - Longitud: entre MIN_WORDS y MAX_WORDS
    - Un único párrafo (sin saltos de línea)
    - Sin listas o numeración
    - Sin menciones de IA/LLM
    - Incluye marcadores de detalle (porque, por ejemplo, aunque...)

    Args:
        text: Texto a evaluar

    Returns:
        Tupla (cumple_todas, diccionario_con_detalles_de_cada_check)
    """
    wc = word_count(text)
    one_paragraph = ("\n" not in text.strip())
    no_lists = not bool(re.search(r"(^|\n)\s*([-*]|\d+\.)\s+", text))
    no_ai_mentions = not bool(re.search(r"\b(IA|inteligencia artificial|LLM|modelo de lenguaje|chatgpt)\b", text, flags=re.IGNORECASE))
    has_detail = bool(re.search(r"\b(porque|por ejemplo|aunque|sin embargo|además)\b", text, flags=re.IGNORECASE))

    # Cumple si TODOS los checks pasan
    ok = (
        MIN_WORDS <= wc <= MAX_WORDS and
        one_paragraph and
        no_lists and
        no_ai_mentions and
        has_detail
    )

    # Información detallada de cada check (para debugging y feedback)
    info = {
        "word_count": wc,
        "one_paragraph": one_paragraph,
        "no_lists": no_lists,
        "no_ai_mentions": no_ai_mentions,
        "has_detail_markers": has_detail,
        "ok": ok
    }
    return ok, info

def eval_score(text: str) -> Tuple[float, Dict[str, Any]]:
    """
    Calcula un score de calidad [0,1] basado en cumplimiento de restricciones.

    Se usa en técnicas feedback-driven y APE para evaluar y clasificar salidas.

    Score = (número de checks que pasan) / (total de checks)

    Args:
        text: Texto a evaluar

    Returns:
        Tupla (score, información_detallada_con_checks)
    """
    ok, info = passes_basic_constraints(text)

    # Ponderación: cada check cumplido suma 1/N puntos
    checks = [
        MIN_WORDS <= info["word_count"] <= MAX_WORDS,
        info["one_paragraph"],
        info["no_lists"],
        info["no_ai_mentions"],
        info["has_detail_markers"],
    ]
    score = sum(checks) / len(checks)  # Score 0-1
    info["score"] = score
    return score, info

def make_topic_block(topic: Optional[str]) -> str:
    """
    Formato lo el bloque de tema (si existe).

    Args:
        topic: Tema (None si no aplica)

    Returns:
        Bloque formateado o string vacío
    """
    if not topic:
        return ""
    return f"Tema principal a enfatizar: {topic}\n"

def build_prompt(p: Dict[str, Any], topic: Optional[str] = None, instruction: Optional[str] = None) -> str:
    """
    Construye un prompt genérico para multi-paso.

    Bloques:
    - Instrucción (customizable)
    - Contexto
    - Tema (si aplica)
    - Restricciones

    Args:
        p: Diccionario de condiciones
        topic: Tema (opcional)
        instruction: Instrucción personalizada (defecto: base_instruction)

    Returns:
        Prompt completo
    """
    instr = instruction or base_instruction()
    return "\n".join([
        instr,
        instance_block(p),
        make_topic_block(topic),
        constraints_block()
    ]).strip()


#### Técnicas

**8. Iterative prompting**

Con esta técnica se evalúa la capacidad del modelo para mejorar progresivamente un conjunto de datos sintéticos mediante un proceso iterativo de generación y reformulación. A partir de un primer conjunto de reseñas generadas, el modelo utiliza esas salidas como contexto para producir nuevas versiones o ejemplos adicionales, incorporando refinamientos graduales en términos de estilo, detalle o cobertura temática.

A diferencia de las técnicas de generación directa (zero-shot, one-shot o few-shot), donde cada reseña se produce de forma independiente, en el iterative prompting las generaciones posteriores están condicionadas por los textos ya producidos. El foco ya no está únicamente en la calidad de una reseña aislada, sino en la evolución progresiva del dataset en su conjunto, permitiendo ampliar o ajustar la distribución de ejemplos a lo largo de múltiples iteraciones.

En este enfoque no es necesario disponer de una función de evaluación explícita: la mejora se produce a través de instrucciones de reformulación, expansión o diversificación aplicadas sobre las salidas previas.

¿Hasta qué punto un LLM es capaz de utilizar sus propias salidas como contexto para generar versiones progresivamente más diversas y coherentes de un conjunto de reseñas a lo largo de varias iteraciones?

¿Qué bloques usa?
* Instrucción base
* Salidas generadas en iteraciones anteriores
* Contexto:
  * producto + perfil del cliente + valoración + tono
  * Tema: explícito.
* Restricciones globales

In [ ]:
# Versiones compactas de funciones para ahorrar tokens en prompts iterativos
# (El modelo tiene límite de contexto, así que necesitamos prompts más concisos)

def constraints_block_short() -> str:
    """
    Versión reducida del bloque de restricciones (ahorra tokens para iteraciones).

    Mantiene esencia pero elimina redundancias.
    """
    return (
        f"Restricciones: {MIN_WORDS}-{MAX_WORDS} palabras; 1 párrafo (sin listas); "
        "incluye detalles concretos de uso; estilo natural y humano de un cliente real; no menciones IA/LLM/sistemas automáticos; "
        "devuelve solo el texto de la reseña"
    )

def instance_block_compact(p: Dict[str, Any], topic: Optional[str] = None) -> str:
    """
    Versión compacta del bloque de condiciones (ahorra tokens).

    Formato: Producto="...", Persona="...", Rating=X/5, Tono="...", Tema="..."

    Args:
        p: Diccionario de condiciones
        topic: Tema (opcional)

    Returns:
        Bloque compacto formateado
    """
    base = f'Producto="{p["product"]}", Persona="{p["persona"]}", Rating={p["rating"]}/5, Tono="{p["tone"]}"'
    if topic:
        base += f', Tema="{topic}"'
    return base

def build_prompt_iterative_base(p: Dict[str, Any], topic: str) -> str:
    """
    ITERACIÓN 0: Genera la reseña inicial.

    Args:
        p: Diccionario de condiciones
        topic: Tema (seleccionado aleatoriamente o asignado)

    Returns:
        Prompt para generación inicial
    """
    return "\n".join([
        base_instruction(),
        f"Condiciones: {instance_block_compact(p, topic=topic)}",
        constraints_block_short()
    ]).strip()

def build_iterative_rewrite_prompt_compact(p: Dict[str, Any], draft_text: str, topic: Optional[str] = None) -> str:
    """
    ITERACIÓN i (i > 0): Reescribe la reseña anterior con mejoras.

    Mejoras enfatizadas:
    - Mayor detalle sobre experiencia de uso
    - Evita repeticiones
    - Mantiene coherencia con rating y tono

    Args:
        p: Diccionario de condiciones
        draft_text: Reseña anterior a mejorar
        topic: Tema (opcional)

    Returns:
        Prompt para reescritura
    """
    return "\n".join([
        "Reescribe la reseña mejorando el nivel de detalle sobre la experiencia de uso, evitando repeticiones y manteniendo coherencia con la valoración y el tono.",
        f"Condiciones: {instance_block_compact(p, topic=topic)}",
        constraints_block_short(),
        "Original:",
        draft_text.strip(),
        "Devuelve solo la reseña final."
    ]).strip()

def generate_iterative_prompting_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    n_iters: int = 1,
    seed: int = 123,
    topics: List[str] = REVIEW_TOPICS,
) -> pd.DataFrame:
    """
    Genera dataset con iterative prompting.

    Para cada muestra:
    1) Genera reseña inicial (ITER 0)
    2) Reescribe n_iters veces para refinar (ITER 1, 2, ...)

    Total de llamadas al modelo: 1 + n_iters por muestra
    Total muestras: 10 condiciones × 5 por condición = 50 reseñas

    Args:
        conditions: Lista de condiciones (reducida a N_CONDITIONS para eficiencia)
        k_per_condition: Muestras por condición
        gen_params: Parámetros del modelo
        n_iters: Número de iteraciones de reescritura
        seed: Semilla para reproducibilidad
        topics: Banco de temas (se elige aleatoriamente)

    Returns:
        DataFrame con las reseñas finales (después de todas las iteraciones)
    """
    rng = random.Random(seed)
    rows = []
    sample_id = 0

    for p in tqdm(conditions, desc="Generando ITERATIVE PROMPTING"):
        for _ in range(k_per_condition):
            # Selecciona tema aleatorio
            topic = rng.choice(topics)

            # ITERACIÓN 0: generación inicial
            prompt_base = build_prompt_iterative_base(p, topic=topic)
            text = chat_generate(prompt_base, gen_params)
            attempts = 1
            last_prompt = prompt_base

            # ITERACIONES 1, 2, ..., n_iters: reescrituras progresivas
            for _it in range(n_iters):
                prompt_rewrite = build_iterative_rewrite_prompt_compact(p, text, topic=topic)
                text = chat_generate(prompt_rewrite, gen_params)
                attempts += 1
                last_prompt = prompt_rewrite

            rows.append(
                make_row(
                    sample_id=sample_id,
                    technique="iterative_prompting",
                    p=p,
                    prompt=last_prompt,            # Último prompt (el de reescritura final)
                    text=text,                      # Texto final después de todas las iteraciones
                    topic=topic,
                    attempts=attempts               # Total de llamadas al modelo
                )
            )
            sample_id += 1

    return pd.DataFrame(rows)


# EJECUCIÓN
print("Iniciando generación de reseñas con ITERATIVE PROMPTING...")
df_iterative_prompting = generate_iterative_prompting_dataset(
    n_iters=1  # 1 iteración de reescritura (2 llamadas totales por muestra)
)

# RESUMEN Y GUARDADO
quick_summary(df_iterative_prompting, "ITERATIVE PROMPTING")
save_outputs(df_iterative_prompting, "iterative_prompting")
print("\nPrimeras 5 filas:")
df_iterative_prompting.head()


**9. Feedback-driven prompting**

Con esta técnica se evalúa la capacidad del modelo para mejorar un conjunto de datos sintéticos utilizando señales explícitas de retroalimentación sobre la calidad de las salidas generadas. Tras producir un conjunto inicial de reseñas, estas son evaluadas mediante criterios externos —por ejemplo, reglas, métricas automáticas, anotaciones humanas o juicios de otro modelo— y esa información se utiliza para filtrar, corregir o regenerar ejemplos.

A diferencia del iterative prompting, donde las mejoras se basan únicamente en la reutilización de las salidas previas como contexto, en el feedback-driven prompting existe una función de evaluación explícita que determina qué ejemplos son aceptables y cuáles deben ser modificados o reemplazados. De este modo, la generación se orienta sistemáticamente hacia una mayor calidad, coherencia o adecuación a los requisitos definidos.

¿Hasta qué punto un LLM es capaz de utilizar señales de evaluación externas para corregir, filtrar y regenerar reseñas, y cómo influye este proceso en la fiabilidad y consistencia del dataset sintético resultante?

¿Qué bloques usa?
* Instrucción base
* Métricas de validación
* Contexto:
  * producto + perfil del cliente + valoración + tono
  * Tema: explícito.
* Restricciones globales

In [ ]:
def build_feedback_fix_prompt(
    p: Dict[str, Any],
    bad_text: str,
    eval_info: Dict[str, Any],
    topic: Optional[str] = None
) -> str:
    """
    Construye prompt para CORRECCIÓN cuando la reseña no cumple restricciones.

    Se le proporciona al modelo:
    1) Descripción de qué falló (con detalles específicos del eval_info)
    2) La reseña anterior incorrecta
    3) Instrucción de corregir

    Args:
        p: Diccionario de condiciones
        bad_text: Reseña actual (no válida)
        eval_info: Diccionario con detalles de qué checks fallaron
        topic: Tema (opcional)

    Returns:
        Prompt para corrección
    """
    # Construye lista de problemas encontrados
    issues = []
    if not (MIN_WORDS <= eval_info["word_count"] <= MAX_WORDS):
        issues.append(f"- Longitud incorrecta: {eval_info['word_count']} palabras (objetivo {MIN_WORDS}-{MAX_WORDS}).")
    if not eval_info["one_paragraph"]:
        issues.append("- Debe ser un único párrafo (sin saltos de línea).")
    if not eval_info["no_lists"]:
        issues.append("- Prohibido usar listas o numeración.")
    if not eval_info["no_ai_mentions"]:
        issues.append("- Prohibido mencionar IA, modelos de lenguaje o sistemas automáticos.")
    if not eval_info["has_detail_markers"]:
        issues.append("- Falta detalle/conectores; añade matices y experiencia concreta (porque, por ejemplo, aunque...).")

    issues_txt = "\n".join(issues) if issues else "- Ninguno."

    return "\n".join([
        "Vas a corregir una reseña para que cumpla estrictamente requisitos.",
        "",
        instance_block(p).strip(),
        "",
        make_topic_block(topic).strip(),
        constraints_block().strip(),
        "",
        "Evaluación externa: la reseña NO cumple por estos motivos:",
        issues_txt,
        "",
        "Reseña a corregir:",
        bad_text.strip(),
        "",
        "Salida: devuelve únicamente el texto final corregido."
    ]).strip()

def generate_feedback_driven_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    max_rounds: int = 4,
    use_topics: bool = True,
    seed: int = 123
) -> pd.DataFrame:
    """
    Genera dataset con feedback-driven prompting.

    Para cada muestra:
    1) Genera reseña
    2) Evalúa con rulesbasadas
    3) Si no cumple, regenera (hasta max_rounds)
    4) Guarda reseña final + información de evaluación

    Total muestras: 10 condiciones × 5 por condición = 50 reseñas

    Args:
        conditions: Lista de condiciones (reducida a N_CONDITIONS)
        k_per_condition: Muestras por condición
        gen_params: Parámetros para generación normal
        max_rounds: Máximo número de iteraciones de corrección
        use_topics: Si True, selecciona temas aleatoriamente
        seed: Semilla para reproducibilidad

    Returns:
        DataFrame con reseñas (y información adicional: score, ok, rounds, etc.)
    """
    random.seed(seed)
    rows = []
    sample_id = 0

    # Usa solo las primeras N_CONDITIONS condiciones
    conditions = conditions[:N_CONDITIONS]

    for p in tqdm(conditions, desc="Generando FEEDBACK-DRIVEN"):
        for _ in range(k_per_condition):
            topic = random.choice(REVIEW_TOPICS) if use_topics else None

            # Generación inicial
            prompt_base = build_prompt(p, topic=topic)
            text = chat_generate(prompt_base, gen_params)
            attempts = 1

            # Evalúa reseña inicial
            score, info = eval_score(text)
            rounds = 0
            last_prompt = prompt_base
            eval_history = []  # Registro de evaluaciones fallidas

            # BUCLE DE CORRECCIÓN: mientras no cumpla y no haya excedido max_rounds
            while (not info["ok"]) and rounds < max_rounds:
                # Guarda info de evaluación anterior para auditoría
                eval_history.append(json.dumps(info, ensure_ascii=False))

                # Solicita corrección con feedback explícito
                last_prompt = build_feedback_fix_prompt(p, text, info, topic=topic)
                # Temperatura ligeramente más baja para que la corrección sea más directa
                text = chat_generate(last_prompt, {**gen_params, "temperature": 0.6})
                attempts += 1

                # Re-evalúa
                score, info = eval_score(text)
                rounds += 1

            # Construye fila final con información de evaluación
            row = make_row(
                sample_id=sample_id,
                technique="feedback_driven_prompting",
                p=p,
                prompt=last_prompt,
                text=text,
                topic=topic,
                attempts=attempts
            )
            # Añade metadatos de evaluación
            row.update({
                "score": score,
                "ok": info.get("ok"),
                "word_count": info.get("word_count"),
                "rounds": rounds,
                "eval_history": "\n---\n".join(eval_history)  # Historial de evaluaciones fallidas
            })

            rows.append(row)
            sample_id += 1

    return pd.DataFrame(rows)

# EJECUCIÓN
print("Iniciando generación de reseñas con FEEDBACK-DRIVEN...")
df_feedback = generate_feedback_driven_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_feedback, "FEEDBACK-DRIVEN")
save_outputs(df_feedback, "feedback_driven_prompting")
print("\nPrimeras 5 filas:")
df_feedback.head()


**10. Automatic Prompt Engineering**

Con esta técnica se evalúa la capacidad de los modelos para optimizar automáticamente las instrucciones (prompts) utilizadas para generar datos sintéticos. En lugar de fijar un único prompt manualmente, el sistema genera múltiples variantes de la instrucción, las evalúa según una métrica de calidad aplicada a los textos producidos y selecciona o refina aquellas que conducen a mejores resultados.

A diferencia del iterative y del feedback-driven prompting, donde el proceso de optimización actúa principalmente sobre los ejemplos generados, en el Automatic Prompt Engineering el objeto principal de optimización es el propio prompt. El proceso se formula como una búsqueda en el espacio de instrucciones: cada prompt se ejecuta para producir un conjunto de reseñas, estas se puntúan según un criterio de calidad y las mejores instrucciones se reutilizan o modifican en iteraciones posteriores.

Este enfoque permite automatizar el diseño de prompts efectivos que, una vez seleccionados, pueden emplearse dentro de pipelines de generación directa, iterativa o con feedback.

¿Hasta qué punto un LLM es capaz de descubrir y refinar automáticamente prompts que maximicen la calidad, coherencia y alineación de los datos sintéticos generados?

¿Qué bloques usa?
* Instrucción base (dinámica, susceptible de modificación)
* Contexto:
  * producto + perfil del cliente + valoración + tono
  * Tema: explícito
* Restricciones globales
* Evaluación automática de los outputs
* Optimización iterativa del prompt

In [ ]:
def build_ape_propose_prompt(task_desc: str, m: int = 8) -> str:
    """
    Solicita al modelo que genere M variantes de instrucciones para la tarea.

    Se le pide que genere prompts (no restricciones), mejorando la tarea de:
    "escribir reseñas realistas de clientes en un solo párrafo"

    Salida esperada: lista numerada de M instrucciones diferentes

    Args:
        task_desc: Descripción de la tarea principal
        m: Número de variantes de instrucción a generar

    Returns:
        Prompt que solicita generar M instrucciones
    """
    return f"""Genera {m} variantes de una instrucción en español para un LLM.
Objetivo: {task_desc}
Requisitos:
- Debe ser una sola frase de instrucción (no incluyas restricciones globales).
- Debe pedir una reseña realista de un cliente.
- No menciones IA/LLMs.
Devuelve SOLO una lista numerada del 1 al {m}, una instrucción por línea.
""".strip()

def parse_numbered_list(text: str, m: int) -> List[str]:
    """
    Extrae instrucciones de una lista numerada del modelo.

    Busca líneas que comienzan con "1.", "2.", etc.
    y extrae el texto después del número.

    Args:
        text: Texto bruto del modelo
        m: Número máximo de instrucciones a extraer

    Returns:
        Lista de instrucciones (hasta M elemento)
    """
    lines = []
    # Busca líneas con formato "N. instrucción"
    for l in text.splitlines():
        if re.search(r"^\s*\d+\.", l):
            # Elimina el número y el punto
            lines.append(re.sub(r"^\s*\d+\.\s*", "", l).strip())

    # Filtra líneas vacías y toma hasta M
    lines = [l for l in lines if l]
    return lines[:m] if lines else []

def generate_ape_dataset(
    conditions: List[Dict[str, Any]] = PRODUCT_CONDITIONS,
    k_per_condition: int = K_PER_CONDITION,
    gen_params: Dict[str, Any] = GEN_DEFAULT,
    m_candidates: int = 8,
    n_trials_per_candidate: int = 1,
    use_topics: bool = True,
    seed: int = 123
) -> pd.DataFrame:
    """
    Genera dataset con Automatic Prompt Engineering.

    Para cada muestra:
    1) Propone M candidatos de instrucción (variantes)
    2) Evalúa N_TRIALS ejemplos con cada candidato
    3) Selecciona el candidato con mejor score promedio
    4) Genera muestra final con la mejor instrucción

    Total muestras: 10 condiciones × 5 por condición = 50 reseñas
    Llamadas al modelo: (1 para proponer) + (M * N_TRIALS para evaluar) + (1 para final)

    Args:
        conditions: Lista de condiciones (reducida a N_CONDITIONS)
        k_per_condition: Muestras por condición
        gen_params: Parámetros para generación
        m_candidates: Número de instrucciones candidatas a generar
        n_trials_per_candidate: Número de intentos para evaluar cada candidato
        use_topics: Si True, selecciona temas aleatoriamente
        seed: Semilla para reproducibilidad

    Returns:
        DataFrame con reseñas finales + información de optimización
    """
    random.seed(seed)
    rows = []
    sample_id = 0

    # Usa solo primeras N_CONDITIONS condiciones
    conditions = conditions[:N_CONDITIONS]

    # Descripción de la tarea para que el modelo genere variantes
    task_desc = "Escribir reseñas de productos de negocio local, en un solo párrafo, naturales y con detalles de experiencia."

    for p in tqdm(conditions, desc="Generando APE"):
        for _ in range(k_per_condition):
            topic = random.choice(REVIEW_TOPICS) if use_topics else None

            # ========== PASO 1: PROPONER CANDIDATOS ==========
            proposal_prompt = build_ape_propose_prompt(task_desc, m=m_candidates)
            proposal_out = chat_generate(proposal_prompt, {**gen_params, "max_tokens": 240, "temperature": 0.9, "top_p": 0.95})
            candidates = parse_numbered_list(proposal_out, m_candidates) or [base_instruction()]
            attempts = 1

            # ========== PASO 2: EVALUAR CANDIDATOS ==========
            cand_audit = []       # Registro de evaluaciones (para auditoría)
            best_instr = None     # Mejor instrucción encontrada
            best_avg = -1.0       # Mejor score promedio

            for instr in candidates:
                scores = []  # Scores de este candidato en sus N_TRIALS

                # Genera N_TRIALS ejemplos para este candidato y evalúa cada uno
                for _t in range(n_trials_per_candidate):
                    prompt_try = build_prompt(p, topic=topic, instruction=instr)
                    text_try = chat_generate(prompt_try, gen_params)
                    attempts += 1

                    # Evalúa reseña generada
                    s, _info = eval_score(text_try)
                    scores.append(s)

                # Score promedio de este candidato
                avg = sum(scores) / len(scores)
                cand_audit.append({"instruction": instr, "avg_score": avg, "scores": scores})

                # Actualiza mejor si este es mejor
                if avg > best_avg:
                    best_avg = avg
                    best_instr = instr

            # Fallback: si no se extrajeron candidatos bien, usa instrucción base
            best_instr = best_instr or base_instruction()

            # ========== PASO 3: GENERAR MUESTRA FINAL ==========
            final_prompt = build_prompt(p, topic=topic, instruction=best_instr)
            final_text = chat_generate(final_prompt, gen_params)
            attempts += 1

            final_score, final_info = eval_score(final_text)

            # Construye fila con información de optimización
            row = make_row(
                sample_id=sample_id,
                technique="automatic_prompt_engineering",
                p=p,
                prompt=final_prompt,
                text=final_text,
                topic=topic,
                attempts=attempts
            )
            # Metadata de APE
            row.update({
                "score": final_score,
                "ok": final_info.get("ok"),
                "word_count": final_info.get("word_count"),
                "best_instruction": best_instr,
                "best_instruction_avg_score": best_avg,
                "candidate_audit_json": json.dumps(cand_audit, ensure_ascii=False),
                "proposal_prompt": proposal_prompt
            })

            rows.append(row)
            sample_id += 1

    return pd.DataFrame(rows)

# EJECUCIÓN
print("Iniciando generación de reseñas con AUTOMATIC PROMPT ENGINEERING...")
df_ape = generate_ape_dataset()

# RESUMEN Y GUARDADO
quick_summary(df_ape, "APE")
save_outputs(df_ape, "automatic_prompt_engineering")
print("\nPrimeras 5 filas:")
df_ape.head()


## Paso 4: Unión del datatset

Se consolidan todos los excel generados en un único dataset para su posterior evaluación en la siguiente fase.

**Salida**: `/content/dataset.xlsx`

In [ ]:
import pandas as pd
from pathlib import Path

# Ruta donde se guardaron los archivos (Colab: /content)
DATA_DIR = Path("/content")

# Columnas a conservar (uniformes entre todas las técnicas)
keep_cols = [
    "technique",
    "condition_id",
    "product",
    "persona",
    "rating",
    "tone",
    "topic",
    "prompt",
    "text",
    "attempts"
]

# Localiza todos los excels generados
excel_files = [
    f for f in DATA_DIR.glob("*.xlsx")
    if not f.name.startswith("~$")  # Excluye archivos temporales
]

print("Excels detectados:")
for f in excel_files:
    print("-", f.name)

dfs = []

# Lee cada excel y extrae solo las columnas compatibles
for f in excel_files:
    df = pd.read_excel(f)

    # Verifica qué columnas tiene este archivo
    available = set(df.columns)
    missing = [c for c in keep_cols if c not in available]

    if missing:
        print(f"{f.name} NO tiene columnas: {missing}")

    # Mantiene solo las columnas que existen
    cols_present = [c for c in keep_cols if c in df.columns]
    df = df[cols_present]

    dfs.append(df)

# Concatena todos los dataframes
df_all = pd.concat(dfs, ignore_index=True)

# Reordena columnas en orden uniforme
df_all = df_all.reindex(columns=keep_cols)

# Guarda el dataset unificado
output_file = "/content/dataset.xlsx"
df_all.to_excel(output_file, index=False)

print(f"\nDataset unificado creado en: {output_file}")
print(f"  Total de filas: {len(df_all)}")
print(f"  Columnas: {list(df_all.columns)}")
